# Sealed camera frames on Jetson 1

Run this notebook in the experiments directory with the Jetson dataset environment as its kernel. Prepare the sealed batch on the trusted workstation using CAMERA.md. A Linux notebook starts the two local protocol roles. The TA decrypts each frame and performs face detection, alignment, cropping, SFace inference and encoding. The results directory must be new for every run.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

experiments = Path.cwd()
assert (experiments / "trials.py").is_file()
# Choose a fresh dealer batch. For two physical parties follow JETSON2.md instead.
batch = experiments / "lab" / "my-camera-batch"
output = experiments.parent / "results" / batch.name
measure_power = True  # The kernel must inherit the verified 100 ms meter on PATH.
trials = json.loads((batch / "drone/trials.json").read_text())
assert trials and all("frame_id" in row for row in trials)
assert (batch / "peer/trials.json").is_file() and not output.exists()

In [ ]:
peer = subprocess.Popen([sys.executable, "trials.py", "peer", "--peer-material", str(batch / "peer")])
try:
    command = [sys.executable, "trials.py", "run", "--drone", str(batch / "drone"),
               "--output", str(output), "--profile-operations"]
    if measure_power:
        command.append("--power")
    subprocess.run(command, check=True)
    if peer.wait(timeout=30):
        raise RuntimeError("The peer exited with an error")
finally:
    if peer.poll() is None:
        peer.terminate()
        peer.wait()

In [ ]:
for setting, row in json.loads((output / "medians.json").read_text()).items():
    print(setting, "trials", row["trials"],
          "median frame to embedding (ms)", row["frame_to_embedding_ms"]["median"],
          "median frame to decision (ms)", row["frame_to_decision_ms"]["median"],
          "median module energy (J)", row["module_frame_to_decision_j"]["median"])